# Notebook 13 — Edit Magnitude as Vulnerability Predictor
**CS 590NN | Amogh | Apr 25 — pure pandas, no GPU, ~30 seconds**

## The signal in existing data

Quick preview on v3 data showed `kl_A_solo` significantly predicts A_displaced:
- When B fails: rho = +0.46, p = 0.04
- When B succeeds: rho = +0.28, p = 0.03

**Pairs where edit A caused MORE drift on the neutral set are pairs where A is MORE vulnerable to subsequent perturbation.** Bigger initial perturbation = less stable edit = more likely to be erased.

## What this notebook does

1. Pool v3 + NB8 data
2. Compute A_displaced = p_new_A_postA - p_new_A_postAB
3. Test predictors of A_displaced with multiple-comparison correction
4. Multivariate regression to find joint predictors
5. Stratify by B success

## Predictions

| Pattern | Conclusion |
|---|---|
| kl_A_solo dominates | Edit magnitude controls vulnerability |
| n_mlp_total dominates | Circuit footprint controls vulnerability |
| Multiple share variance | Heterogeneous mechanism |
| Nothing significant after Bonferroni | Original signal was noise |

### 13.0 Setup

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path
from datetime import datetime

RESULTS_DIR = Path('results_nb13'); RESULTS_DIR.mkdir(exist_ok=True)
FIG_DIR = RESULTS_DIR / 'figures'; FIG_DIR.mkdir(exist_ok=True)
pd.set_option('display.float_format', '{:.3f}'.format)
print('Setup done.')

### 13.1 Load data

In [ ]:
from google.colab import files
print('Upload sequential_edit_rows_v3_*.json AND beta_kl_ablation_*.json (NB8)')
uploaded = files.upload()
v3_file = next((k for k in uploaded if 'sequential_edit_rows' in k), None)
nb8_file = next((k for k in uploaded if 'beta_kl_ablation' in k or 'beta_ablation' in k), None)

with open(v3_file) as f: v3 = pd.DataFrame([r for r in json.load(f)['rows'] if r.get('status')=='ok'])
with open(nb8_file) as f: nb8 = pd.DataFrame([r for r in json.load(f)['rows'] if r.get('status')=='ok'])

print(f'v3: {len(v3)} rows, nb8: {len(nb8)} rows')

### 13.2 Compute A_displaced

In [ ]:
for d in (v3, nb8):
    d['A_displaced'] = (d['p_new_A_postA'] - d['p_new_A_postAB']).clip(lower=0)
    d['B_installed'] = d['p_new_B_postAB']

v3_clean = v3[v3['p_new_A_postA'] > 0.5].copy()
nb8_clean = nb8[nb8['p_new_A_postA'] > 0.5].copy()
print(f'Clean v3: {len(v3_clean)}, clean nb8: {len(nb8_clean)}')

v3_b_failed = v3_clean[v3_clean['B_installed'] < 0.5]
v3_b_succ   = v3_clean[v3_clean['B_installed'] >= 0.5]
print(f'\nv3 split: B failed={len(v3_b_failed)}, B succeeded={len(v3_b_succ)}')
print(f'  Mean A_displaced when B failed:    {v3_b_failed["A_displaced"].mean():.3f}')
print(f'  Mean A_displaced when B succeeded: {v3_b_succ["A_displaced"].mean():.3f}')

### 13.3 Univariate predictors with Bonferroni

In [ ]:
candidate_cols = ['kl_A_solo', 'kl_B_solo', 'kl_postA',
                  'n_mlp_A', 'n_mlp_B', 'n_attn_A', 'n_attn_B',
                  'p_new_A_solo', 'p_new_A_postA', 'p_new_B_solo']

def test_predictors(df, label):
    print(f'\n=== {label} (n={len(df)}) ===')
    results = []
    for col in candidate_cols:
        if col not in df.columns: continue
        rho, p = stats.spearmanr(df[col], df['A_displaced'])
        results.append((col, rho, p))
    n_tests = len(results)
    print(f'  variable        rho       p     p_bonf   sig')
    for col, rho, p in results:
        p_bonf = min(1.0, p * n_tests)
        sig = '***' if p_bonf<0.001 else ('**' if p_bonf<0.01 else ('*' if p_bonf<0.05 else ''))
        print(f'  {col:>14}  {rho:+.3f}  {p:.4f}  {p_bonf:.4f}  {sig}')
    return results

r_all = test_predictors(v3_clean, 'POOLED v3 (clean)')
r_b_failed = test_predictors(v3_b_failed, 'when B FAILED (baseline displacement)')
r_b_succ   = test_predictors(v3_b_succ, 'when B SUCCEEDED (active displacement)')

### 13.4 Multivariate regression

In [ ]:
from numpy.linalg import lstsq

def multivar_ols(df, cols):
    sub = df.dropna(subset=cols + ['A_displaced'])
    if len(sub) < len(cols) + 5: return None
    X = sub[cols].values
    X = np.column_stack([np.ones(len(X)), X])
    y = sub['A_displaced'].values
    beta, _, _, _ = lstsq(X, y, rcond=None)
    y_pred = X @ beta
    ss_res = ((y - y_pred)**2).sum()
    ss_tot = ((y - y.mean())**2).sum()
    r2 = 1 - ss_res/ss_tot if ss_tot > 0 else 0.0
    return beta, r2, len(sub)

v3_clean['n_mlp_total'] = v3_clean['n_mlp_A'] + v3_clean['n_mlp_B']
v3_clean['n_attn_total'] = v3_clean['n_attn_A'] + v3_clean['n_attn_B']

models = {
    'kl_A_solo only': ['kl_A_solo'],
    'kl_A + B_installed': ['kl_A_solo', 'B_installed'],
    'kl_A + n_mlp_total': ['kl_A_solo', 'n_mlp_total'],
    'full (kl_A + n_mlp + B)': ['kl_A_solo', 'n_mlp_total', 'B_installed'],
    'full + n_attn': ['kl_A_solo', 'n_mlp_total', 'n_attn_total', 'B_installed'],
}

print('=== Multivariate OLS ===')
for name, cols in models.items():
    res = multivar_ols(v3_clean, cols)
    if res is None:
        print(f'  {name}: insufficient data')
        continue
    beta, r2, n = res
    coef_str = ', '.join([f'{c}={beta[i+1]:+.3f}' for i, c in enumerate(cols)])
    print(f'  {name:>30}: R2={r2:.3f}  intercept={beta[0]:+.3f}  n={n}')
    print(f'  {" "*32}{coef_str}')

### 13.5 Per-method check

In [ ]:
print('=== kl_A_solo -> A_displaced by method ===')
for m in v3_clean['method'].unique():
    sub = v3_clean[v3_clean['method']==m]
    if len(sub) < 5: continue
    rho, p = stats.spearmanr(sub['kl_A_solo'], sub['A_displaced'])
    sig = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else ''))
    print(f'  {m:>11}  rho={rho:+.3f}  p={p:.4f}  n={len(sub)}  {sig}')

print('\n=== by condition ===')
for c in v3_clean['condition'].unique():
    sub = v3_clean[v3_clean['condition']==c]
    if len(sub) < 5: continue
    rho, p = stats.spearmanr(sub['kl_A_solo'], sub['A_displaced'])
    sig = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else ''))
    print(f'  {c:>10}  rho={rho:+.3f}  p={p:.4f}  n={len(sub)}  {sig}')

### 13.6 Money plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
for label, sub, color in [
    (f'B failed (n={len(v3_b_failed)})', v3_b_failed, '#888888'),
    (f'B succeeded (n={len(v3_b_succ)})', v3_b_succ, '#cc3333'),
]:
    ax.scatter(sub['kl_A_solo'], sub['A_displaced'], color=color, s=55, alpha=0.7,
               edgecolors='black', lw=0.4, label=label)
rho_all, p_all = stats.spearmanr(v3_clean['kl_A_solo'], v3_clean['A_displaced'])
ax.set_xlabel('kl_A_solo (drift caused by edit A alone)')
ax.set_ylabel('A_displaced')
ax.set_title(f'Edit magnitude vs vulnerability\npooled rho={rho_all:+.3f}, p={p_all:.4f}')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
v3_clean['kl_A_quartile'] = pd.qcut(v3_clean['kl_A_solo'], 4, labels=['Q1 low', 'Q2', 'Q3', 'Q4 high'])
groups = [v3_clean[v3_clean['kl_A_quartile']==q]['A_displaced'].values for q in v3_clean['kl_A_quartile'].cat.categories]
bp = ax.boxplot(groups, labels=v3_clean['kl_A_quartile'].cat.categories, patch_artist=True, widths=0.6)
for patch in bp['boxes']: patch.set_facecolor('#aaccee')
ax.set_xlabel('kl_A_solo quartile')
ax.set_ylabel('A_displaced')
ax.set_title('A displacement by edit-A drift magnitude')
ax.grid(alpha=0.3, axis='y')
fig.tight_layout(); fig.savefig(FIG_DIR / 'fig1_kl_A_solo_vs_displaced.png', dpi=140); plt.show()

### 13.7 Save

In [ ]:
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
rho_pooled, p_pooled = stats.spearmanr(v3_clean['kl_A_solo'], v3_clean['A_displaced'])
rho_b_failed, p_b_failed = stats.spearmanr(v3_b_failed['kl_A_solo'], v3_b_failed['A_displaced']) if len(v3_b_failed) > 5 else (np.nan, np.nan)
rho_b_succ, p_b_succ = stats.spearmanr(v3_b_succ['kl_A_solo'], v3_b_succ['A_displaced']) if len(v3_b_succ) > 5 else (np.nan, np.nan)

summary = {
    'notebook': 'Notebook 13 - Edit magnitude predictor',
    'timestamp': ts, 'n_clean_v3': len(v3_clean),
    'kl_A_solo_vs_A_displaced': {
        'pooled': {'rho': float(rho_pooled), 'p': float(p_pooled)},
        'when_B_failed': {'rho': float(rho_b_failed), 'p': float(p_b_failed), 'n': len(v3_b_failed)},
        'when_B_succeeded': {'rho': float(rho_b_succ), 'p': float(p_b_succ), 'n': len(v3_b_succ)},
    },
    'verdict': ('kl_A_solo is significant predictor' if p_pooled < 0.05 else 'no significant predictor'),
}
with open(RESULTS_DIR / f'summary_nb13_{ts}.json', 'w') as f: json.dump(summary, f, indent=2)
v3_clean.to_csv(RESULTS_DIR / f'analysis_df_{ts}.csv', index=False)
print(json.dumps(summary, indent=2))

import shutil
bundle = f'nb13_results_{ts}'
shutil.make_archive(bundle, 'zip', RESULTS_DIR)
from google.colab import files
files.download(f'{bundle}.zip')